In [1]:
import numpy as np
import matplotlib.pyplot as plt
from time import time
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from math import sqrt
from numba import njit


In [2]:
def generate_data(n, dimensions=2):
    """Generate n random points in [0, 1]^dimensions."""
    return np.random.uniform (0, 1, (n, dimensions))

In [63]:
p = np.random.uniform(0, 1, (10, 2))  # 10 points in P
q = np.random.uniform(0, 1, (10, 2))  # 10 points in Q

In [3]:
#np.random.seed(4)
n = 5
p = generate_data(n)
q = generate_data(n)

# Baseline O(n^2)

In [4]:

def baseline_alg(points1, points2):
    n = len(points1)
    unique_x = np.concatenate([points1[:, 0], points2[:, 0]])
    unique_y = np.concatenate([points1[:, 1], points2[:, 1]])
    p_or_q = np.concatenate([np.ones(n), np.zeros(n)])


    max_diff = 0
    sorted_indices_y = np.argsort(unique_y);

    for i in range(len(unique_x) - 1):
        pq_diff = 0 

        #print("reset pq_diff", pq_diff)

        for j in range(len(unique_y) - 1):

            #print(i,j,sorted_indices_y[j],unique_x[sorted_indices_y[j]],unique_x[i])

            #check if x coordinate of j violates x threshold from i
            # if true, then this point is out of the x range, and move to next point in y order
            if(unique_x[i] < unique_x[sorted_indices_y[j]]): 
                continue;
            

            #print("compare:", sorted_indices_y[j], n)

            # if point j is in P:  sorted_y[j] indexes into the unique_y array, first ones in P, others in Q
            if (p_or_q[sorted_indices_y[j]] == 1) :
                pq_diff += 1
            else:
                pq_diff -= 1 

            #print(sorted_indices_y[j], pq_diff)

            if abs(pq_diff) > max_diff:
                max_diff = abs(pq_diff)

    # print(max_diff)

    float_diff = float(max_diff)
    float_n = float(n)

    return float_diff / float_n


baseline_alg(p, q)



0.4

In [5]:
from numba import njit
import numpy as np

@njit
def baseline_alg(points1, points2):
    n = len(points1)
    
    # Create combined arrays manually
    unique_x = np.zeros(2 * n)
    unique_y = np.zeros(2 * n)
    p_or_q = np.zeros(2 * n)

    # Populate unique_x, unique_y, and p_or_q for both points1 and points2
    for i in range(n):
        unique_x[i] = points1[i, 0]
        unique_y[i] = points1[i, 1]
        p_or_q[i] = 1  # Points in P

        unique_x[n + i] = points2[i, 0]
        unique_y[n + i] = points2[i, 1]
        p_or_q[n + i] = 0  # Points in Q

    # Initialize variables
    max_diff = 0
    sorted_indices_y = np.argsort(unique_y)

    # Outer loop: Sweep over max x
    for i in range(2 * n):
        pq_diff = 0
        # Inner loop: Sweep over max y
        for j in range(2 * n):
            idx = sorted_indices_y[j]
            if unique_x[i] < unique_x[idx]:  # Check x threshold
                continue

            # Adjust pq_diff based on whether the point belongs to P or Q
            if p_or_q[idx] == 1:
                pq_diff += 1
            else:
                pq_diff -= 1

            # Update max_diff if necessary
            max_diff = max(max_diff, abs(pq_diff))

    return max_diff / n


# Optimized Algorithm 

In [6]:
def optimized_function(points1, points2):
    n = len(points2)
    grid_size = 2 * int(sqrt(n))
    
    # Initialize counters for each cell to zero
    counts_p = np.zeros((grid_size, grid_size), dtype=int)
    counts_q = np.zeros((grid_size, grid_size), dtype=int)

    # combined_sorted_x = np.sort(np.concatenate([points1[:, 0], points2[:, 0]]))
    # combined_sorted_y = np.sort(np.concatenate([points1[:, 1], points2[:, 1]]))
    combined_sorted_x_indices = np.argsort(np.concatenate([points1[:, 0], points2[:, 0]]))
    combined_sorted_y_indices = np.argsort(np.concatenate([points1[:, 1], points2[:, 1]]))

    # # Step 3: Select evenly spaced grid boundaries based on the combined sorted coordinates
    # selected_x = combined_sorted_x[::len(combined_sorted_x) // (grid_size + 1)]
    # selected_y = combined_sorted_y[::len(combined_sorted_y) // (grid_size + 1)]

    num_points = len(combined_sorted_x_indices)
    step = num_points // (grid_size + 1)
    selected_x = np.concatenate([points1[:, 0], points2[:, 0]])[combined_sorted_x_indices][::step]
    selected_y = np.concatenate([points1[:, 1], points2[:, 1]])[combined_sorted_y_indices][::step]


    # Assign indices to points
    # @njit
    # def assign_indices(points, selected_x, selected_y):
    #     indices = []
    #     for x, y in points:
    #         i = np.searchsorted(selected_x, x, side='right') - 1
    #         j = np.searchsorted(selected_y, y, side='right') - 1
    #         indices.append((min(i, grid_size - 1), min(j, grid_size - 1)))  # Ensure indices are within bounds
    #     return indices
    


    # indices_p = assign_indices(points1, selected_x, selected_y, grid_size)
    # indices_q = assign_indices(points2, selected_x, selected_y, grid_size)

    # # Increment counters based on indices
    # for i, j in indices_p:
    #     counts_p[i, j] += 1
    # for i, j in indices_q:
    #     counts_q[i, j] += 1


    
    def increment_counts(points, selected_x, selected_y, counts, grid_size):
        for x, y in points:
            i = np.searchsorted(selected_x, x, side='right') - 1
            j = np.searchsorted(selected_y, y, side='right') - 1
            counts[min(i, grid_size - 1), min(j, grid_size - 1)] += 1

    increment_counts(points1, selected_x, selected_y, counts_p, grid_size)
    increment_counts(points2, selected_x, selected_y, counts_q, grid_size)


    
    def cumulative_count(counts):
        rows, cols = counts.shape
        Cx = np.zeros_like(counts)

        for i in range(rows):
            for j in range(cols):
                if i == 0:  # Special case for the first row
                    Cx[i, j] = counts[i, j-1] if j > 0 else counts[i, j]
                elif j == 0:  # Special case for the first column
                    Cx[i, j] = Cx[i-1, j] + counts[i, j]
                else:
                    Cx[i, j] = Cx[i-1, j] + Cx[i, j-1] - Cx[i-1, j-1] + counts[i, j]

        return Cx

    # Compute cumulative counts
    Cp = cumulative_count(counts_p)
    Cq = cumulative_count(counts_q)

    # Calculate differences
    difference = np.abs(Cp - Cq)
    max_diff = np.max(difference)

    return float(max_diff) / float(n)

# Example usage
max_diff = optimized_function(p, q)
print(f"max diff: {max_diff}")



max diff: 0.4


In [77]:
# max_diff= optimized_function(p, q)

# print("Cp:")
# print(Cp)
# print("\nCq:")
# print(Cq)
# print("\nMaximum absolute difference:")
# print(max_diff/n)
# print("max diff position", max_diff_position)


# Evaluation

In [15]:
# Evaluation with averages over 10 runs
n_values = np.array([128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072], dtype=int)

baseline_runtimes = []
optimized_runtimes = []
baseline_errors = []
optimized_errors = []

num_runs = 20  # Number of runs for averaging

for n in n_values:
    baseline_runtime_sum = 0
    optimized_runtime_sum = 0
    baseline_error_sum = 0
    optimized_error_sum = 0
    
    for _ in range(num_runs):
        p = generate_data(n)
        q = generate_data(n)
        
        # Baseline algorithm
        start_time = time()
        baseline_error = baseline_alg(p, q)
        baseline_runtime_sum += (time() - start_time)
        baseline_error_sum += baseline_error
        
        # Optimized algorithm
        start_time = time()
        optimized_error = optimized_function(p, q)
        optimized_runtime_sum += (time() - start_time)
        optimized_error_sum += optimized_error
    
    # Store the averages
    baseline_runtimes.append(baseline_runtime_sum / num_runs)
    optimized_runtimes.append(optimized_runtime_sum / num_runs)
    baseline_errors.append(baseline_error_sum / num_runs)
    optimized_errors.append(optimized_error_sum / num_runs)


plt.figure(figsize=(12, 5))

# Plot 1: Runtime vs n
plt.subplot(1, 2, 1)
plt.plot(n_values, baseline_runtimes, 's--', label='Baseline', color='orange')
plt.plot(n_values, optimized_runtimes, 'o-', label='Our Algo', color='blue')
plt.xscale('linear')
plt.yscale('linear')
plt.xlabel('n (# samples)')
plt.ylabel('Runtime (s)')
plt.title('Runtime vs n')
plt.legend()

# Plot 2: Error vs n
plt.subplot(1, 2, 2)
plt.plot(n_values, baseline_errors, 's--', label='Baseline', color='orange')
plt.plot(n_values, optimized_errors, 'o-', label='Our Algo', color='blue')
plt.xscale('linear')
plt.yscale('linear')
plt.xlabel('n (# samples)')
plt.ylabel('Observed Error')
plt.title('Observed Error vs n')
plt.legend()
plt.tight_layout()
plt.savefig('results.pdf', format='pdf')
plt.show()

In [131]:
def measure_runtime(optimized_function, n_samples, num_runs=100):
    runtimes = []
    for _ in range(num_runs):
        points1 = generate_data(n_samples)
        points2 = generate_data(n_samples)
        start_time = time()
        optimized_function(points1, points2)
        runtimes.append(time() - start_time)
    avg_runtime = sum(runtimes) / num_runs
    return avg_runtime

# Measure the average runtime for n=10
n_samples = 10
num_runs = 100
avg_runtime = measure_runtime(optimized_function, n_samples, num_runs)

print(f"Average runtime for n={n_samples} over {num_runs} runs: {avg_runtime:.6f} seconds")

Average runtime for n=10 over 100 runs: 0.000202 seconds
